## CRF - baseline

In [ ]:
%pip install -r ../requirements.txt

In [ ]:
# fix on the imports
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [ ]:
# parameters

NO_SAMPLES : int   = 50000
SEED:      int   = 42
N_FOLDS:   int   = 3

CV = 3
N_ITER = 20
N_JOBS = -1

In [ ]:

# preprocessing data with commmon project method
from data_preparation.preprocess_data import preprocess_data

df = preprocess_data()[:NO_SAMPLES]

In [ ]:
print(df['target'][0])
print(df['input'][0])

In [ ]:
from data_preparation.labels import align_labels

In [ ]:
test_adddress = df['input'][0]
test_parsed = df['target'][0]

tokens, labels = align_labels(test_adddress, test_parsed)
print("Tokens:", tokens)
print("Labels:", labels)

In [ ]:
import regex as re

def word_features(tokens: list[str], i: int) -> dict:
    word = tokens[i]
    features = {
        "bias":           1.0,
        "word.lower":     word.lower(),
        "word.isupper":   word.isupper(),
        "word.istitle":   word.istitle(),
        "word.isdigit":   word.isdigit(),
        "word.len":       len(word),
        "prefix2":        word[:2].lower(),
        "suffix2":        word[-2:].lower(),
        "is_num_pattern": bool(re.match(r'^\d+[a-zA-Z]?$', word)),
        "is_postal":      bool(re.match(r'\b[A-Za-z0-9][A-Za-z0-9\- ]{2,9}[A-Za-z0-9]\b', word)),
        "is_cc":          bool(re.match(r'^[A-Z]{2}$', word)),
        "is_first":         i == 0,
        "is_last":          i == len(tokens) - 1,
        "position":         i,
        "position_rel":     round(i / max(len(tokens) - 1, 1), 2),
    }

    # Context window ±2
    for offset, prefix in [(-2,"m2"), (-1,"m1"), (1,"p1"), (2,"p2")]:
        j = i + offset
        if 0 <= j < len(tokens):
            features[f"{prefix}:word.lower"] = tokens[j].lower()
            features[f"{prefix}:isdigit"]    = tokens[j].isdigit()
            features[f"{prefix}:istitle"]    = tokens[j].istitle()
        else:
            features[f"{prefix}:BOS_EOS"] = True

    return features

In [ ]:
test_tokens = ['Ep', 'Epovska', '81', 'Lubihjë', 'XK', 'Epërme', 'Epërme']
for i in range(len(test_tokens)):
    print(f"Features for '{test_tokens[i]}':")
    print(word_features(test_tokens, i))

In [ ]:
import pandas as pd

def prepare_split(split_df: pd.DataFrame):
    tokenized_addresses = []
    aligned_labels = []

    for _, row in split_df.iterrows():
        if row['target'].get('country') == 'Taiwan':
            continue

        raw_address = row['input']
        target_dict = row['target']
        tokens, labels = align_labels(raw_address, target_dict)

        if not tokens:
            continue

        tokenized_addresses.append(tokens)
        aligned_labels.append(labels)

    return tokenized_addresses, aligned_labels

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats
from data_preparation.preprocess_data import make_splits
import regex as re
import sklearn_crfsuite

print("Tuning hyperparameters on fold 0...")

first_split = next(make_splits(df))
train_tokens, train_labels = prepare_split(first_split['train'])
val_tokens,   val_labels   = prepare_split(first_split['val'])

X_train = [[word_features(list(t), i) for i in range(len(t))] for t in train_tokens]
X_val   = [[word_features(list(t), i) for i in range(len(t))] for t in val_tokens]
y_train = [list(l) for l in train_labels]
y_val   = [list(l) for l in val_labels]

X_tune = X_train + X_val
y_tune = y_train + y_val

crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    max_iterations=200,
    all_possible_transitions=True,
)

param_distributions = {
    "c1": stats.expon(scale=0.5),  
    "c2": stats.expon(scale=0.05),
}

search = RandomizedSearchCV(
    crf,
    param_distributions,
    cv=CV,
    n_iter=N_ITER,
    scoring="f1_weighted", 
    n_jobs=N_JOBS,
    verbose=1,
    random_state=SEED,
)
search.fit(X_tune, y_tune)

best_c1 = search.best_params_["c1"]
best_c2 = search.best_params_["c2"]
print(f"Best params → c1: {best_c1:.4f} | c2: {best_c2:.4f}")

In [ ]:
from data_preparation.labels import bio_to_dict

In [ ]:
from data_preparation.preprocess_data import evaluate_predictions

for fold_num, split in enumerate(make_splits(df)):
    print(f"\n═══════════ Fold {fold_num} ═══════════")

    train_tokens, train_labels = prepare_split(split['train'])
    test_tokens,  test_labels  = prepare_split(split['test'])

    X_train = [[word_features(list(t), i) for i in range(len(t))] for t in train_tokens]
    X_test  = [[word_features(list(t), i) for i in range(len(t))] for t in test_tokens]
    y_train = [list(l) for l in train_labels]
    y_test  = [list(l) for l in test_labels]

    crf = sklearn_crfsuite.CRF(
        algorithm="lbfgs", c1=best_c1, c2=best_c2,
        max_iterations=200, all_possible_transitions=True,
    )
    crf.fit(X_train, y_train)
    y_crf_pred = crf.predict(X_test)

    label_names = sorted([l for l in crf.classes_ if l != "O"])

    gold_dicts     = list(split['test']['target'])
    crf_pred_dicts = [bio_to_dict(t, l) for t, l in zip(test_tokens, y_crf_pred)]
    crf_results    = evaluate_predictions(gold_dicts, crf_pred_dicts)

    print("\n── CRF metrics ──")
    print(f"Exact match:       {crf_results['exact_match']:.3f}")
    print(f"Overall item acc:  {crf_results['overall_item_accuracy']:.3f}")
    for field, acc in crf_results['per_field_accuracy'].items():
        print(f"  {field:<15} {acc:.3f}")